# ZelusBench: Sustained Attention (Long) — Depth 15-18

Does performance collapse across deep dependency chains?

**Independent variable**: chain depth ∈ [15, 18]
- Noise ratio fixed at 1.5x (num_points = round(depth * 1.5))
- transform_prob=0.1, dim=3 fixed
- POSITION queries only, query_min_depth = depth - 2
- 10 seeds per depth = 20 scenarios

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import time

from zelusbench.scenarios.config import ScenarioConfig, QueryType
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response

DEPTHS = [15, 18]
NOISE_RATIO = 1.5
SEEDS = 10
TOTAL = len(DEPTHS) * SEEDS
print(f"ZelusBench Sustained Attention (Long) — Depth 15-18")
print(f"Depths: {DEPTHS} | Noise ratio: {NOISE_RATIO}x | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
def _prompt_with_retry(llm, prompt, attempts=4):
    for attempt in range(attempts):
        try:
            return llm.prompt(prompt)
        except Exception as e:
            if attempt == attempts - 1:
                raise
            wait = 2 ** attempt
            print(f"    !! API error ({type(e).__name__}), retry {attempt+1}/{attempts-1} in {wait}s")
            time.sleep(wait)

@kbench.task(name="zelusbench_sustained_long")
def zelusbench_sustained_long(llm) -> float:
    _start = time.time()
    print(f"Running {TOTAL} scenarios...")
    print("=" * 60)
    all_scenario_avgs = []
    depth_scores = {}
    scenario_num = 0
    for depth in DEPTHS:
        num_pts = round(depth * NOISE_RATIO)
        depth_scores[depth] = []
        for i in range(SEEDS):
            scenario_num += 1
            bg_rng = random.Random(i * 7919)
            cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
                "dim": 3,
                "min_chain_depth": depth, "max_chain_depth": depth,
                "num_points": num_pts,
                "transform_prob": 0.1,
                "query_types": [QueryType.POSITION],
                "query_min_depth": max(1, depth - 2),
                "num_queries": 3,
                "seed": depth * 1000 + i,
            })
            scenario = ScenarioGenerator(cfg).generate(f"sustained_long_{depth}_{i}")
            response = _prompt_with_retry(llm, scenario.prompt)
            scores = score_response(response, scenario)
            avg = float(np.mean([s.score for s in scores]))
            depth_scores[depth].append(avg)
            all_scenario_avgs.append(avg)
            print(f"  [{scenario_num}/{TOTAL}] depth={depth} pts={num_pts} avg={avg:.2f} | lb={cfg.leaf_bias}")
        print(f"  >> Depth {depth} mean: {float(np.mean(depth_scores[depth])):.3f}")
    print("\n" + "=" * 60)
    for depth in DEPTHS:
        num_pts = round(depth * NOISE_RATIO)
        avg = round(float(np.mean(depth_scores[depth])), 3)
        print(f"  depth={depth:2d}  pts={num_pts:2d}  accuracy={avg:.3f}")
        kbench.assertions.assert_true(
            avg > 0.0,
            expectation=f"Sustained long depth={depth} ({num_pts} pts): accuracy={avg:.3f}"
        )
    overall = round(float(np.mean(all_scenario_avgs)), 3)
    print(f"\nScore: {overall:.3f} | {len(all_scenario_avgs)} scenarios | {time.time()-_start:.1f}s")
    return overall

zelusbench_sustained_long.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_sustained_long